# 06 | Comparacao das quatro campanhas padronizadas

**Papel nesta entrega:** Principal: comparacao de saidas salvas

**Origem:** https://colab.research.google.com/drive/1PA4hiukRZeh2ThNsnu6h35bJfO_tQlq4

Copia de publicacao organizada em 7 out. 2026. Saidas, contadores e metadados de sessao foram removidos; isso nao significa que a campanha original nao tenha sido executada. Nenhum resultado novo foi produzido na curadoria. O registro de proveniencia relaciona codigo, fonte e alteracoes de configuracao.

Esta versao historica da comparacao e anterior a auditoria MEDINA/R4. Ela reproduz as comparacoes de R1-R3, os modelos e os controles entao disponiveis. O complemento de R4 e agregado pelo caderno 10, sem alterar o snapshot original. Sua frase historica sobre MEDINA/R4 nao representa o estado atual do TCC.

# Comparacao padronizada entre IMTM, FMP, MEDINA e USC

Este caderno compara as quatro execucoes didaticas ja concluidas. Ele **nao treina modelos, nao redefine candidatos e nao abre MEDINA/R4**. As referencias sao locais: o mesmo metodo nao produz automaticamente a mesma unidade biologica.

IMTM e FMP de 03/10/2026 substituem apenas a entrada do novo comparativo; o primeiro comparativo permanece preservado. As hipoteses biologicas exigem perfis com sinal, imagens e, quando pertinente, validacao experimental.

**Como usar:** mantenha o pacote de fontes ao lado do caderno e execute todas as celulas. O calculo completo ja foi realizado localmente com essas fontes. A celula de comparacao gera uma pasta nova e nao modifica arquivos anteriores.

## 1. Fontes, bibliotecas e integridade

Nao instalamos ou substituimos bibliotecas no meio da sessao. Sao necessarios NumPy, pandas, SciPy e Matplotlib, normalmente disponiveis no Colab. O SHA-256 identifica o pacote exato utilizado; nomes de arquivo iguais nao bastam para garantir a mesma versao.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, zipfile, shutil, platform
import numpy as np
import pandas as pd
import scipy
import matplotlib
from IPython.display import display, Markdown, Image
from google.colab import drive
drive.mount("/content/drive")
BASE = Path('/content/drive/MyDrive/mba_tcc/TCC_CellPainting_cadernos_repositorio/04_fontes_e_referencia/recursos/fontes')
PACOTE_FONTES = BASE / "fontes_comparacao_padronizada_4locais.zip"
HASH_FONTES = '3e15afe9572e4b883710cd63898861f536d7a46fa9da7f1de01ed02e9e402902'
assert PACOTE_FONTES.is_file(), f"Pacote ausente: {PACOTE_FONTES}"
assert hashlib.sha256(PACOTE_FONTES.read_bytes()).hexdigest() == HASH_FONTES, "O pacote nao corresponde a esta versao."
RUN = BASE / "execucoes" / datetime.now(timezone.utc).strftime("comparacao_%Y%m%dT%H%M%S_%fZ")
RUN.mkdir(parents=True, exist_ok=False)
FONTES = RUN / "fontes"
FONTES.mkdir()
with zipfile.ZipFile(PACOTE_FONTES) as z:
    for item in z.infolist():
        target = (FONTES / item.filename).resolve()
        if not target.is_relative_to(FONTES.resolve()):
            raise ValueError("Caminho inseguro no ZIP")
    z.extractall(FONTES)
print("Saida desta comparacao:", RUN)
print("Versoes do comparador:", platform.python_version(), np.__version__, pd.__version__, scipy.__version__, matplotlib.__version__)

## 2. O que sera conferido

A unidade de correspondencia e composto e dose numerica, sem juntar automaticamente controles com doses divergentes. Sao recontados os criterios para todas as condicoes, as metricas de todas as predições externas e as proporcoes de DMSO usando apenas controles avaliaveis.

O escore A continua sendo RMS/P95 do respectivo local. A comparacao de Spearman descreve ordenacao de magnitudes locais, **nao identidade da assinatura celular**. As seis direcoes reutilizam condicoes e nao representam seis experimentos adicionais. Os modelos mantem seus treinamentos locais, sem transferencia entre sitios.

## 3. Codigo integral

As funcoes abaixo leem as tabelas e fazem auditorias, juncoes e graficos. Nao existe chamada de treinamento. As funcoes finais verificam os 31 parametros cientificos, as versoes declaradas e a diferenca entre as campanhas historicas e as padronizadas.

In [ ]:
"""Comparacao retrospectiva de quatro execucoes locais ja concluidas.
Sem treinamento, redefinicao de referencias, exclusao por qualidade ou leitura de MEDINA R4.
As bibliotecas necessarias sao apenas NumPy, pandas, SciPy e matplotlib.
"""
from pathlib import Path
from itertools import combinations
from decimal import Decimal, InvalidOperation
from datetime import datetime, timezone
import json, hashlib, warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SITES=['IMTM','FMP','MEDINA','USC']
BLOCKS=['AGP','DNA','ER','Mito','Morfologia_Cells','Morfologia_Cyto','Morfologia_Nuc']
REPS=['R1','R2','R3']
KEY=['composto','dose_chave']
RENAME={'Metadata_Plate_ID':'placa','Metadata_EOS':'composto','Metadata_Concentration':'dose',
        'completa_3rep_7blocos':'avaliavel','prioridade_R1_R3_original':'candidato','cosseno_mito_min':'cosseno_min'}
MODEL_RENAME={'Copiar_A_Mito':'Copiar_escore','Calibracao_A_Mito':'Calibracao_escore'}

def require(condition,message):
    if not bool(condition): raise ValueError(message)

def digest(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def boolean(s):
    if pd.api.types.is_bool_dtype(s): return s.fillna(False)
    values=s.fillna('false').astype(str).str.lower()
    require(values.isin(['true','false','1','0']).all(),'Valor booleano nao reconhecido.')
    return values.isin(['true','1'])

def dose_key(x):
    try:
        d=Decimal(str(x));require(d.is_finite(),'Dose nao finita.')
        return format(d.normalize(),'f')
    except (InvalidOperation,ValueError):
        raise ValueError('Dose nao numerica: conferir a fonte, nao adivinhar unidades.')

def csv(path): return pd.read_csv(path,encoding='utf-8-sig')

def save(df,out,name):
    df.to_csv(out/'tabelas'/f'{name}.csv',index=False)
    return df

def markdown(df,digits=4):
    a=df.copy()
    for c in a:
        if pd.api.types.is_float_dtype(a[c]): a[c]=a[c].map(lambda v:'NA' if pd.isna(v) else f'{v:.{digits}f}')
    a=a.fillna('NA').astype(str)
    return '| '+' | '.join(a.columns)+' |\n| '+' | '.join(['---']*len(a.columns))+' |\n'+'\n'.join('| '+' | '.join(r)+' |' for r in a.to_numpy())

def check_sources(root):
    manifest=json.loads((root/'manifesto_fontes.json').read_text())
    for item in manifest:
        p=(root/item['arquivo']).resolve();require(root.resolve() in p.parents,'Caminho de fonte inseguro.')
        require(p.is_file() and digest(p)==item['sha256'],'Fonte alterada/ausente: '+item['arquivo'])
    # MEDINA R4 nao deve estar no pacote. Metadados de disponibilidade sao permitidos.
    require(not list((root/'MEDINA').glob('15_*')),'Nao incluir resultados da reserva MEDINA.')
    return manifest

def load_conditions(root):
    paths={site:'06_todas_condicoes.csv' for site in SITES}
    frames=[]
    for site,file in paths.items():
        q=csv(root/site/file).rename(columns=RENAME)
        q['site']=site;q['dose_chave']=q.dose.map(dose_key)
        for c in ['avaliavel','candidato']:q[c]=boolean(q[c])
        require(not q.duplicated(['placa']+KEY).any(),'Condicao duplicada em '+site)
        scorecols=[f'A_{b}_{r}' for b in BLOCKS for r in REPS]
        finite=np.isfinite(q[scorecols].to_numpy(float)).all(axis=1)
        q['n_replicas_Mito_acima']=q[[f'A_Mito_{r}' for r in REPS]].gt(1).sum(axis=1)
        othercols=[c for c in scorecols if not c.startswith('A_Mito_')]
        q['outros_max']=q[othercols].max(axis=1)
        q['passa_alvo_recontado']=q.n_replicas_Mito_acima.ge(2)
        q['passa_outros_recontado']=finite&q[othercols].le(1).all(axis=1)
        q['passa_cosseno_recontado']=q.cosseno_min.gt(0)
        rebuilt=q.avaliavel&finite&q.passa_alvo_recontado&q.passa_outros_recontado&q.passa_cosseno_recontado
        require(rebuilt.equals(q.candidato),'Regra nao reproduzida em '+site)
        q['regra_reproduzida']=rebuilt.eq(q.candidato)
        q['motivos_comparacao']=[('nao_avaliavel' if not r.avaliavel else ';'.join([k for k,b in [('magnitude_Mito',r.passa_alvo_recontado),('outros_blocos',r.passa_outros_recontado),('direcao',r.passa_cosseno_recontado)] if not b]) or 'atende_regra') for r in q.itertuples()]
        q['codigo_EOS']=q.composto.str.fullmatch(r'EOS[0-9]+')
        frames.append(q[['site','placa','composto','dose','dose_chave','avaliavel','candidato','cosseno_min','codigo_EOS','n_replicas_Mito_acima','outros_max','passa_alvo_recontado','passa_outros_recontado','passa_cosseno_recontado','motivos_comparacao','regra_reproduzida']+scorecols+[f'A_{b}_mediana' for b in BLOCKS]])
    allc=pd.concat(frames,ignore_index=True)
    eos=allc[allc.codigo_EOS].copy()
    require(not eos.duplicated(['site']+KEY).any(),'EOS e dose com varias placas: resolver sem media automatica.')
    return allc,eos

def load_reserve(root):
    rows=[]
    for site,path,status in [(x,'15_avaliacao_candidatos_reserva.csv','estado_reserva') for x in SITES if x != 'MEDINA']:
        q=csv(root/site/path).rename(columns=RENAME)
        target='A_alvo_reserva'
        cos='cosseno_min_reserva'
        suffix='reserva'
        other=[f'A_{b}_{suffix}' for b in BLOCKS if b!='Mito']
        finite=np.isfinite(q[[target,cos]+other].to_numpy(float)).all(axis=1)
        repeated=finite&q[target].gt(1)&q[other].le(1).all(axis=1)&q[cos].gt(0)
        require(repeated.eq(q[status].eq('padrao_repetido')).all(),'Repeticao R4 nao reproduzida: '+site)
        rows.append(pd.DataFrame({'site':site,'composto':q.composto,'dose_chave':q.dose.map(dose_key),'placa':q.placa,'status_R4':q[status], 'A_Mito_R4':q[target],'cosseno_R4':q[cos], 'outros_max_R4':q[other].max(axis=1),'motivos_R4':q.motivos,'margem_Mito_R4':q[target]-1}))
    return pd.concat(rows,ignore_index=True)

def metrics(y,p):
    y=np.asarray(y,float);p=np.asarray(p,float)
    require(np.isfinite(y).all() and np.isfinite(p).all(),'Predicao log nao finita.')
    ya,pa=np.expm1(y),np.expm1(p)
    def r2(a,b):
        den=np.square(a-a.mean()).sum();return 1-np.square(a-b).sum()/den if den else np.nan
    return dict(MAE_log=float(abs(y-p).mean()),RMSE_log=float(np.sqrt(np.square(y-p).mean())),R2_log=float(r2(y,p)),Spearman=float(spearmanr(y,p).statistic),MAE_A=float(abs(ya-pa).mean()),RMSE_A=float(np.sqrt(np.square(ya-pa).mean())),R2_A=float(r2(ya,pa)))

def audit_predictions(root,out):
    perdir=[];audit=[];extreme=[];global_extreme=[];foldaudits=[]
    for site in SITES:
        q=csv(root/site/'12_predicoes_externas.csv')
        cond=csv(root/site/'06_todas_condicoes.csv'); cond=cond[boolean(cond.avaliavel)].set_index('id_condicao')
        parts=csv(root/site/'11_particoes_compostos.csv').set_index('id_condicao')
        require(parts.index.is_unique and set(parts.index)==set(cond.index),'Particoes/coorte divergentes: '+site)
        require(not q.duplicated(['id_condicao','direcao','modelo']).any(),'OOF duplicada: '+site)
        require(set(q.modelo)=={'Mediana_treino','Copiar_escore','Calibracao_escore','Ridge_4_escores','Ridge_medidas','XGBoost'},'Modelos divergentes.')
        require(q.groupby('composto').fold.nunique().eq(1).all(),'Codigo em multiplos folds.')
        require(q.fold.between(1,5).all(),'Fold invalido.')
        require(q.origem.isin(REPS).all() and q.destino.isin(REPS).all() and q.origem.ne(q.destino).all(),'Replicas invalidas.')
        require(np.isfinite(q[['y_log','pred_log','A_observado','A_predito']]).all().all(),'Predicoes nao finitas: nao excluir silenciosamente.')
        require(np.allclose(np.log1p(q.A_observado),q.y_log,rtol=1e-10,atol=1e-10),'log alvo divergente.')
        require(np.allclose(np.expm1(q.pred_log),q.A_predito,rtol=1e-9,atol=1e-9),'Inversa divergente.')
        require(len(q)==len(cond)*36,'Coorte OOF incompleta.')
        for (d,m),g in q.groupby(['direcao','modelo']):
            require(set(g.id_condicao)==set(cond.index),'Conjunto avaliado diferente entre comparadores.')
            c=cond.loc[g.id_condicao]; ps=parts.loc[g.id_condicao]
            require(np.array_equal(g.composto.to_numpy(),c.composto.to_numpy()),'Identidade OOF divergente.')
            require(np.array_equal(g.fold.to_numpy(),ps.fold.to_numpy()),'Fold diverge da particao salva.')
            require(np.array_equal(g.placa.to_numpy(),c.placa.to_numpy()) and np.allclose(g.dose,c.dose),'Placa/dose OOF divergente.')
            src=g.origem.iloc[0];dst=g.destino.iloc[0]
            require(np.allclose(g.A_observado,c['A_Mito_'+dst],rtol=1e-9,atol=1e-9),'Alvo diferente do escore congelado.')
            require(np.allclose(g.A_origem,c['A_Mito_'+src],rtol=1e-9,atol=1e-9),'Escore de origem divergente.')
            perdir.append(dict(site=site,direcao=d,modelo=m,n=len(g),**metrics(g.y_log,g.pred_log)))
            errs=abs(g.A_predito-g.A_observado);mx=g.loc[errs.idxmax()]
            extreme.append(dict(site=site,direcao=d,modelo=m,composto=mx.composto,placa=mx.placa,fold=mx.fold,A_observado=mx.A_observado,A_predito=mx.A_predito,fracao_SAE_maior_erro=errs.max()/errs.sum(),n=len(g)))
        for m,g in q.groupby('modelo'):
            errs=abs(g.A_predito-g.A_observado);mx=g.loc[errs.idxmax()]
            global_extreme.append(dict(site=site,modelo=m,composto=mx.composto,placa=mx.placa,direcao=mx.direcao,fold=mx.fold,y_log=mx.y_log,pred_log=mx.pred_log,A_observado=mx.A_observado,A_predito=mx.A_predito,fracao_SAE_global=errs.max()/errs.sum(),n=len(g)))
        audit.append(dict(site=site,n_linhas_predicoes_auditadas=len(q),n_modelos_comparadores=q.modelo.nunique(),max_erro_transformacao_alvo=float(abs(np.log1p(q.A_observado)-q.y_log).max()),max_erro_inversa=float(abs(np.expm1(q.pred_log)-q.A_predito).max()),escopo='Chaves, particoes salvas e valores; sem reexecutar ajustes'))
        aa=csv(root/site/'12_auditoria_modelos.csv');aa['site']=site
        require(len(aa)==30,'Auditoria local nao contem 30 ajustes.')
        foldaudits.append(aa)
    pdirection=pd.DataFrame(perdir)
    summary=pdirection.groupby(['site','modelo'],as_index=False)[list(metrics([0,1],[0,1]))].mean()
    for site in SITES:
        saved=csv(root/site/'12_metricas_resumo.csv').set_index('modelo'); cur=summary[summary.site.eq(site)].set_index('modelo')
        for col in list(metrics([0,1],[0,1])):
            require(np.allclose(cur.loc[saved.index,col],saved[col],rtol=1e-7,atol=1e-7,equal_nan=True),'Metrica nao reproduzida: '+site+' '+col)
    save(pd.DataFrame(global_extreme),out,'extremos_globais_sem_exclusao')
    save(pd.concat(foldaudits,ignore_index=True),out,'auditorias_locais_modelos_sem_recalculo')
    summary['origem_metrica']='recalculada das predições salvas'
    return save(summary,out,'modelos_resumo'),save(pdirection,out,'modelos_direcoes_recontadas'),save(pd.DataFrame(audit),out,'auditoria_predicoes'),save(pd.DataFrame(extreme),out,'extremos_por_modelo_direcao')

def check_dmso(root,out):
    frames=[];checks=[]
    for site in SITES:
        refs=csv(root/site/'05_referencias_congeladas.csv')
        scores=csv(root/site/'05_escores_por_poco.csv')
        controls=scores[scores.composto.eq('DMSO')]
        require(not scores.duplicated(['placa','replica','poco']).any(),'Poco duplicado: '+site)
        for row in refs.itertuples():
            q=controls[controls.placa.eq(row.placa)]['RMS_'+row.bloco]
            finite=q[np.isfinite(q)]
            p95=float(finite.quantile(.95,interpolation='linear'))
            require(len(finite)==row.n_DMSO and np.isclose(p95,row.P95,rtol=1e-10,atol=1e-10),'P95 nao reproduzido: '+site+row.placa+row.bloco)
            ii=scores.placa.eq(row.placa)
            require(np.allclose(scores.loc[ii,'A_'+row.bloco],scores.loc[ii,'RMS_'+row.bloco]/row.P95,rtol=1e-9,atol=1e-9,equal_nan=True),'RMS/P95 divergente.')
            checks.append(dict(site=site,placa=row.placa,bloco=row.bloco,P95_salvo=row.P95,P95_recontado=p95,erro_abs=abs(p95-row.P95),n_DMSO=len(finite)))
        stages=[('05_DMSO.csv','desenvolvimento',scores)]
        if site!='MEDINA':stages.append(('15_DMSO_reserva.csv','reserva',csv(root/site/'15_escores_reserva.csv')))
        for file,stage,sc in stages:
            tab=csv(root/site/file);tab['site']=site;tab['etapa']=stage
            for rr in tab.itertuples():
                a=sc[sc.placa.eq(rr.placa)&sc.replica.eq(rr.replica)&sc.composto.eq('DMSO')]['A_'+rr.bloco]
                finite=a[np.isfinite(a)];n=len(finite);above=int(finite.gt(1).sum())
                require(len(a)==rr.n_DMSO and n==rr.n_avaliaveis and above==rr.n_acima,'DMSO contagem divergente.')
                expected=100*above/n if n else np.nan
                require(np.isclose(expected,rr.taxa_pct,equal_nan=True),'Denominador DMSO divergente.')
            frames.append(tab)
    save(pd.DataFrame(checks),out,'referencias_P95_recontadas')
    return save(pd.concat(frames,ignore_index=True),out,'DMSO_placa_replica')

def comparison(root,out):
    out=Path(out);require(not out.exists(),'Use pasta nova para preservar execucoes.')
    (out/'tabelas').mkdir(parents=True);(out/'figuras').mkdir()
    manifest=check_sources(root)
    allc,eos=load_conditions(root);reserve=load_reserve(root)
    save(allc,out,'condicoes_quatro_locais');save(reserve,out,'avaliacao_R4_reanalises_e_USC')
    labels=csv(root/'USC/14_identidades_toda_coorte.csv')[['composto','name','status_estrutura','gene_name_resumo','moa_resumo']]
    require(labels.composto.is_unique,'Identidades duplicadas.')
    eos=eos.merge(labels,on='composto',how='left',validate='many_to_one')
    info=json.loads((root/'inventario_execucoes.json').read_text())
    summary=[]
    for site,g in allc.groupby('site',sort=False):
        rr=reserve[reserve.site.eq(site)]
        summary.append({'site':site,**info[site],'condicoes_total':len(g),'condicoes_completas':int(g.avaliavel.sum()),'EOS_completos':int((g.avaliavel&g.codigo_EOS).sum()),'candidatos':int(g.candidato.sum()),'avaliados_R4':len(rr),'repetiram_R4':int(rr.status_R4.eq('padrao_repetido').sum()) if len(rr) else np.nan,'taxa_R4_pct':100*rr.status_R4.eq('padrao_repetido').mean() if len(rr) else np.nan})
    summary=save(pd.DataFrame(summary),out,'resumo_quatro_locais')
    # Somente os mesmos codigos e doses avaliaveis nos quatro locais; nenhum filtro pelo resultado.
    pairsets=[set(map(tuple,g.loc[g.avaliavel,KEY].to_numpy())) for _,g in eos.groupby('site')]
    shared=set.intersection(*pairsets)
    common=eos[[tuple(k) in shared for k in eos[KEY].to_numpy()]].copy()
    require(len(common)==len(shared)*4,'Coorte comum ambigua.')
    save(common,out,'coorte_comum_4_locais')
    r=[]
    for a,b in combinations(SITES,2):
        ma=common[common.site.eq(a)].merge(common[common.site.eq(b)],on=KEY,suffixes=('_a','_b'),validate='one_to_one')
        require(ma.placa_a.eq(ma.placa_b).all(),'Codigo mudou de placa; rever estratificacao.')
        for pl,g in [('todas',ma),*list(ma.groupby('placa_a'))]:
            rho=float(spearmanr(g.A_Mito_mediana_a,g.A_Mito_mediana_b).statistic)
            inter=int((g.candidato_a & g.candidato_b).sum());union=int((g.candidato_a | g.candidato_b).sum())
            r.append({'site_a':a,'site_b':b,'placa':pl,'n_condicoes':len(g),'Spearman_A_Mito_local':rho,'intersecao_candidatos':inter,'uniao_candidatos':union,'Jaccard_candidatos':inter/union if union else np.nan})
    pairs=save(pd.DataFrame(r),out,'concordancia_pares_e_placas')
    # Matrizes de todos os candidatos de qualquer local, nao apenas os que concordam.
    candidatekeys=set(map(tuple,eos.loc[eos.candidato,KEY].to_numpy()))
    candidatecross=eos[[tuple(k) in candidatekeys for k in eos[KEY].to_numpy()]].merge(reserve.drop(columns=['placa']),on=['site']+KEY,how='left',validate='one_to_one')
    candidatecross['status_R4']=candidatecross.status_R4.fillna('nao_avaliado_como_candidato_neste_local')
    save(candidatecross,out,'candidatos_uniao_entre_locais')
    state=common.pivot(index=KEY,columns='site',values='candidato')
    save(state.reset_index(),out,'matriz_prioridade_coorte_comum')
    mag=common.pivot(index=KEY,columns='site',values='passa_alvo_recontado')
    save(mag.reset_index().assign(n_locais_Mito_acima=mag.sum(axis=1).to_numpy()),out,'matriz_excedencia_Mito_coorte_comum')
    plate=[];flow=[]
    for site,g in eos.groupby('site'):
        for pl,z in g[g.avaliavel].groupby('placa'):
            plate.append(dict(site=site,placa=pl,n=len(z),n_candidatos=int(z.candidato.sum()),pct_candidatos=100*z.candidato.mean(),n_mito2=int(z.passa_alvo_recontado.sum())))
        flow.append(dict(site=site,completas=int(g.avaliavel.sum()),Mito_2ou3=int((g.avaliavel&g.passa_alvo_recontado).sum()),Mito_e_direcao=int((g.avaliavel&g.passa_alvo_recontado&g.passa_cosseno_recontado).sum()),Mito_e_outros=int((g.avaliavel&g.passa_alvo_recontado&g.passa_outros_recontado).sum()),regra_total=int(g.candidato.sum())))
    plate=save(pd.DataFrame(plate),out,'candidatos_por_placa');save(pd.DataFrame(flow),out,'funil_criterios_por_local')
    models,directions,predaudit,extremes=audit_predictions(root,out)
    dmso=check_dmso(root,out)
    benefits=[]
    for site,g in models.groupby('site'):
        g=g.set_index('modelo');v=float(g.loc['XGBoost','MAE_log']);cp=float(g.loc['Copiar_escore','MAE_log'])
        best=g.drop(index='XGBoost').MAE_log.idxmin();bv=float(g.loc[best,'MAE_log'])
        benefits.append(dict(site=site,MAE_XGBoost=v,MAE_copia=cp,ganho_vs_copia_pct=100*(cp-v)/cp,melhor_referencia=best,MAE_melhor_referencia=bv,ganho_vs_melhor_pct=100*(bv-v)/bv,R2_log=float(g.loc['XGBoost','R2_log']),R2_A=float(g.loc['XGBoost','R2_A'])))
    benefits=save(pd.DataFrame(benefits),out,'ganho_XGBoost_vs_referencias')
    # Casos predefinidos por razoes explicitas, depois da modelagem. Nao constituem novo filtro.
    reasons={'EOS100315':'Alda-1: alvo mitocondrial caracterizado em literatura primaria; caso biologico pos-hoc', 'EOS102190':'Glaucine: unica intersecao das listas locais nesta execucao', 'EOS100964':'MM-102: candidato FMP repetido em R4 e mecanismo nuclear anotado', 'EOS102355':'TCID: candidato IMTM repetido com maior margem original', 'EOS102419':'DHODH-IN-17: registro mitocondrial com criterio completo nao repetido em R4', 'EOS100147':'Hispidulin: repeticao USC, para contraste de respostas entre locais'}
    cases=eos[eos.composto.isin(reasons)].merge(reserve.drop(columns=['placa']),on=['site']+KEY,how='left',validate='one_to_one')
    cases['razao_estudo_caso']=cases.composto.map(reasons)
    save(cases,out,'casos_biologicos_quatro_locais')
    # Mantemos as assinaturas descritivas separadas de SHAP.
    for site in SITES:
        counts=csv(root/site/'02_contagens_pocos.csv');domin=csv(root/site/'05_dominancia_por_poco.csv')
        small=counts[counts.n_objetos.lt(20)].copy();save(small,out,site+'_pocos_menos20_sem_exclusao')
        dom=domin[domin.bloco.eq('Mito')].copy();dom['quadrado_maior_pct']=100*dom.fracao_quadratica_maior_medida
        save(dom[dom.fracao_quadratica_maior_medida.gt(.8)],out,site+'_dominancia_Mito_alerta')
        selected=counts[counts.composto.isin(candidatekeys and {x[0] for x in candidatekeys} or set())].copy()
        save(selected,out,site+'_pocos_candidatos_para_inspecao')
    for name in ['08_assinaturas_medidas.csv','08_assinaturas_familias.csv','15_assinaturas_reserva_medidas.csv','15_assinaturas_reserva_familias.csv']:
        z=csv(root/'USC'/name);save(z[z.composto.isin(reasons)],out,'USC_casos_'+name.replace('.csv',''))
    # Escalas SHAP nao sao comparadas como efeito causal; ranking por local e familia somente.
    shaprows=[]
    for site,path,val,ncol in [(x,'13_SHAP_global.csv','SHAP_abs_medio','n_modelos') for x in SITES]:
        f=csv(root/site/path);f['site']=site;f=f.rename(columns={val:'SHAP_abs',ncol:'modelos_selecionada'})
        f=f.sort_values('SHAP_abs',ascending=False);f['rank_local']=np.arange(1,len(f)+1)
        shaprows.append(f[['site','medida','familia','bloco','SHAP_abs','modelos_selecionada','rank_local']])
    shap=pd.concat(shaprows,ignore_index=True);save(shap,out,'SHAP_rankings_quatro_locais')
    # As quatro tabelas quantitativas mantem os valores originais. Escalas dependem de modelos distintos.
    sf=shap.groupby(['site','bloco','familia'],as_index=False).agg(soma_abs=('SHAP_abs','sum'),n_medidas=('medida','size'),media_modelos_selecionada=('modelos_selecionada','mean'))
    sf['fracao_atribuicoes_local']=sf.soma_abs/sf.groupby('site').soma_abs.transform('sum')
    save(sf,out,'SHAP_familias_blocos_quatro_locais')
    tests={'n_condicoes_comuns':len(shared),'n_uniao_candidatos':len(candidatekeys),'regras_reproduzidas_todas':bool(allc.regra_reproduzida.all()),'n_locais_com_dados':eos.site.nunique(),'MEDINA_R4_lida':False,'reajustou_modelos':False,'fontes_inalteradas':True}
    # Figuras, uma por arquivo; cores padrao do matplotlib, sem alteracao de dados.
    legends={}
    def figdone(fig,name,caption):
        fig.tight_layout();fig.savefig(out/'figuras'/f'{name}.png',dpi=180,bbox_inches='tight');fig.savefig(out/'figuras'/f'{name}.svg',bbox_inches='tight');plt.close(fig);legends[name]=caption
    fig,ax=plt.subplots(figsize=(8,4.4));ss=summary.set_index('site').loc[SITES];ax.bar(SITES,ss.candidatos);ax.set_ylabel('Condicoes prioritarias');ax.set_title('Listas locais: mesma regra, referencias distintas')
    for k,v in enumerate(ss.candidatos):ax.text(k,v+.8,str(v),ha='center')
    figdone(fig,'C01_prioridades','Contagem local, nao acuracia; a ausencia de prioridade nao implica ausencia de resposta mitocondrial.')
    corr=common.pivot(index=KEY,columns='site',values='A_Mito_mediana').reindex(columns=SITES).corr(method='spearman')
    fig,ax=plt.subplots(figsize=(6.2,5.3));im=ax.imshow(corr,vmin=-1,vmax=1);ax.set_xticks(range(4),SITES);ax.set_yticks(range(4),SITES)
    for i in range(4):
        for j in range(4):ax.text(j,i,f'{corr.iloc[i,j]:.3f}',ha='center',va='center')
    fig.colorbar(im,ax=ax,label='Spearman');ax.set_title(f'Posicao relativa de A_Mito: {len(shared)} condicoes comuns')
    figdone(fig,'C02_concordancia','Escores com caracteristicas e P95 locais distintos. Concordancia de ordenacao nao e concordancia de perfis completos nem transferencia de modelo.')
    fig,ax=plt.subplots(figsize=(9,4.6))
    for site,g in plate.groupby('site'):ax.plot(g.placa,g.pct_candidatos,marker='o',label=site)
    ax.set_ylabel('Prioridades / EOS completos da placa (%)');ax.set_title('Distribuicao das prioridades por placa');ax.legend()
    figdone(fig,'C03_placas','Os denominadores sao os EOS completos de cada placa. Composicao quimica e efeitos experimentais nao estao causalmente separados.')
    fig,ax=plt.subplots(figsize=(8,4.7));bm=benefits.set_index('site').loc[SITES];ax.bar(SITES,bm.ganho_vs_copia_pct);ax.set_ylabel('Reducao do MAE log frente a copia (%)');ax.set_title('Ganho dentro de cada local, sem transferencia entre locais')
    figdone(fig,'C04_ganho_modelo','Ganhos calculados separadamente nas mesmas observacoes de cada local. Nao comparar R2 como percentual de acertos. Nao houve novo ajuste.')
    fig,ax=plt.subplots(figsize=(8,4.6));z=dmso[(dmso.etapa=='reserva')&(dmso.bloco=='Mito')]
    for site,g in z.groupby('site'):
        if 'taxa_pct' in g:ax.plot(g.placa,g.taxa_pct,marker='o',label=site)
    ax.axhline(5,linestyle='--');ax.set_ylabel('DMSO acima do limite original (%)');ax.set_title('R4: controles avaliaveis; MEDINA nao aberta');ax.legend()
    figdone(fig,'C05_controles_R4','As proporcoes usam apenas escores avaliaveis. Valores ausentes nao sao zero; FMP possui duas placas nao avaliaveis.')
    fig,ax=plt.subplots(figsize=(8,4.6))
    for code in ['EOS100315','EOS102190','EOS100964','EOS102355']:
        q=cases[cases.composto.eq(code)].set_index('site').reindex(SITES)
        ax.plot(SITES,q.A_Mito_mediana,marker='o',label=str(q.name.iloc[0]))
    ax.axhline(1,linestyle='--');ax.set_ylabel('Mediana local de A_Mito, R1-R3');ax.set_title('Casos: magnitude local nao equivale a seletividade');ax.legend()
    figdone(fig,'C06_casos','Apresentacao pos-hoc com criterios declarados. Medidas/P95 diferem entre locais; linhas conectam resumos, nao trajetorias celulares.')
    fig,ax=plt.subplots(figsize=(9,5));z=pairs[pairs.placa.ne('todas')]
    for (a,b),g in z.groupby(['site_a','site_b']):ax.plot(g.placa,g.Spearman_A_Mito_local,marker='o',label=a+' / '+b)
    ax.set_ylabel('Spearman entre escores locais');ax.set_title('Concordancia dos mesmos compostos, estratificada por placa');ax.legend(fontsize=8,ncol=2)
    figdone(fig,'C07_concordancia_placas','Somente a coorte compartilhada pelos quatro locais, sem selecao por resposta. Correlacao nao demonstra mecanismo comum.')
    fig,ax=plt.subplots(figsize=(8,4.5));z=reserve[reserve.site.eq('USC')&reserve.status_R4.eq('padrao_repetido')].sort_values('margem_Mito_R4');ax.bar(range(len(z)),100*z.margem_Mito_R4);ax.set_xlabel(f'{len(z)} condicoes, ordenadas somente para exibicao');ax.set_ylabel('Margem de A_Mito acima de 1 em R4 (%)');ax.set_title('USC: atendimento binario e distancia do limite')
    figdone(fig,'C08_margens_USC','Todas as condicoes repetidas foram preservadas. O corte de 5% de margem mencionado no relatorio e apenas uma descricao, nao um filtro novo.')
    (out/'legendas.json').write_text(json.dumps(legends,ensure_ascii=False,indent=2))
    (out/'conferencias.json').write_text(json.dumps(tests,ensure_ascii=False,indent=2))
    check_sources(root)
    # Relatorio automatico, mantendo interpretacao biologica em um arquivo curado separado.
    report='# Comparacao retrospectiva entre quatro locais\n\n'+markdown(summary[['site','condicoes_total','condicoes_completas','EOS_completos','medidas_mito','candidatos','avaliados_R4','repetiram_R4']])+ '\n\n## Coorte comum e concordancia\n\n'+f'Foram comparados {len(shared)} pares unicos EOS-dose avaliaveis nos quatro locais. Os controles nomeados foram separados, sem consertar doses automaticamente. A uniao das listas locais tem {len(candidatekeys)} codigos/condicoes. A regra foi reproduzida em todas as linhas exportadas.\n\n'+markdown(pairs[pairs.placa.eq('todas')][['site_a','site_b','n_condicoes','Spearman_A_Mito_local','intersecao_candidatos','uniao_candidatos']])+ '\n\n## Predicao\n\n'+markdown(benefits)+ '\n\n## Casos biologicos: dados antes da interpretacao\n\n'+markdown(cases[['site','name','A_Mito_mediana','outros_max','cosseno_min','candidato','status_R4']])+ '\n\n## Alcance das conferencias\n\n'+markdown(predaudit)+ '\n\nRecontadas as regras locais, as chaves e as transformacoes das predições salvas. Nao foram rerodados os modelos nem a extracao das imagens. Os seis comparadores de cada local foram recontados a partir das predições salvas; as métricas impressas IMTM foram conferidas com tolerância de arredondamento. As auditorias de SHAP dos ajustes locais permanecem como fontes, sem recalculo de atribuicoes. MEDINA R4 nao foi lida. Nao foram realizadas novas normalizacoes, exclusoes, testes de enriquecimento ou ajustes de limites.\n\n## Limites da comparacao\n\nOs resultados sao de ajustes locais, com caracteristicas selecionadas e P95 diferentes. As correlacoes comparam ordenacoes de magnitudes locais, nao uma representacao fenotipica comum. As quatro campanhas usam o nucleo didatico. Os parametros cientificos e as versoes declaradas foram confrontados; adaptadores e fontes locais sao distintos. A padronizacao nao torna identicas as escalas ou as amostras de treinamento. A coorte e os alvos precederam os folds, de forma que a avaliacao preditiva continua condicional. Nao se demonstra transferencia de modelos, generalizacao para classes quimicas ou mecanismo biologico. Mais locais nao garantem mais concordancia.\n'
    ncommon4=int(mag.all(axis=1).sum())
    up=reserve[reserve.site.eq('USC')].copy()
    up['repete']=up.status_R4.eq('padrao_repetido')
    upstats=up.groupby('placa',as_index=False).agg(candidatos=('composto','size'),repetidos=('repete','sum'))
    near=int((up.repete & up.A_Mito_R4.le(1.05)).sum())
    qc=dmso[(dmso.etapa=='reserva') & (dmso.bloco=='Mito')].groupby('site',as_index=False).agg(n_DMSO=('n_DMSO','sum'),avaliaveis=('n_avaliaveis','sum'),acima=('n_acima','sum'))
    qc['nao_avaliaveis']=qc.n_DMSO-qc.avaliaveis
    qc['taxa_entre_avaliaveis_pct']=100*qc.acima/qc.avaliaveis
    save(upstats,out,'USC_candidatos_repeticao_por_placa')
    save(qc,out,'DMSO_R4_resumo_por_local')
    report += f"\n## Magnitude compartilhada e distribuicao USC\n\n{ncommon4} de {len(shared)} condicoes comuns excederam Mito em pelo menos duas replicas nos quatro locais. Sao {100*ncommon4/len(shared):.2f}% da coorte comum; isso nao aplica os outros criterios nem cria candidatos.\n\n"+markdown(upstats)+f"\n\nEntre os {int(up.repete.sum())} casos repetidos USC, {near} tiveram 1 < A_Mito_R4 <= 1.05. A faixa e descritiva, sem alterar a regra ou escolher cortes. A concentracao por placa nao separa composicao quimica e efeitos experimentais.\n\n## DMSO de R4: denominadores explicitos\n\n"+markdown(qc)+f"\n\nRecontadas {int(predaudit.n_linhas_predicoes_auditadas.sum())} predições e {len(allc)} linhas de condições. As matrizes brutas de todas as imagens e os 120 ajustes XGBoost nao foram reexecutados. MEDINA R4 permanece fora da analise; taxa de repeticao sem candidatos e nao aplicavel, nao zero por cento.\n"
    (out/'relatorio_comparativo.md').write_text(report,encoding='utf-8')
    return {'summary':summary,'pairs':pairs,'benefits':benefits,'cases':cases,'tests':tests,'out':out}



SCIENTIFIC_KEYS=['desenvolvimento','reservada','controles','bloco_alvo','canais','morfologia','min_controles','fator_MAD','epsilon','correlacao_maxima','quantil_referencia','min_replicas_ativas','cosseno_minimo','alerta_objetos_abaixo','alerta_dominancia_acima','min_objetos_inclusao','seed','n_bootstrap','executar_agrupamento','k_candidatos','n_subamostras','executar_modelos','folds_externos','folds_internos','max_arvores','paciencia','ridge_alphas','bootstrap_modelos','min_fracao_finita_X','min_IQR_X','xgb']

def audit_protocols(root,out):
    prot={s:json.loads((root/s/'protocolo.json').read_text()) for s in SITES}
    rows=[]
    for k in SCIENTIFIC_KEYS:
        values={s:json.dumps(prot[s]['config'][k],sort_keys=True,ensure_ascii=False) for s in SITES}
        require(len(set(values.values()))==1,'Parametro cientifico divergente: '+k)
        rows.append(dict(parametro=k,**values,identico=True))
    save(pd.DataFrame(rows),out,'parametros_cientificos_entre_locais')
    v=pd.DataFrame([dict(site=s,**prot[s]['versoes']) for s in SITES])
    for col in v.columns.drop('site'):require(v[col].nunique(dropna=False)==1,'Versao declarada divergente: '+col)
    save(v,out,'versoes_declaradas_nas_execucoes')
    hashes=pd.DataFrame([dict(site=s,nucleo_completo=prot[s]['config'].get('nucleo_sha256'),nucleo_calculos=prot[s]['config'].get('nucleo_calculos_sha256'),reserva_previamente_consultada=prot[s]['config']['reserva_ja_consultada']) for s in SITES])
    save(hashes,out,'identificadores_implementacao_e_reserva')
    # Full hash includes ingestion adapters; equality is not imposed across different schemas.
    return dict(parametros_comparados=len(rows),parametros_cientificos_iguais=True,versoes_declaradas_iguais=True,hashes_completos_iguais=hashes.nucleo_completo.nunique()==1)

def compare_historical(root,out):
    old=csv(root/'historico/condicoes_quatro_locais.csv');cur=csv(out/'tabelas/condicoes_quatro_locais.csv');rr=[]
    for site in ['IMTM','FMP']:
        a=cur[cur.site.eq(site)];b=old[old.site.eq(site)]
        m=a.merge(b,on=['placa','composto','dose'],suffixes=('_novo','_historico'),validate='one_to_one',how='outer',indicator=True)
        require(m._merge.eq('both').all(),'Condicoes historicas ausentes/adicionadas.')
        cols=[c for c in a if c.startswith('A_')]
        x=m[[c+'_novo' for c in cols]].to_numpy(float);y=m[[c+'_historico' for c in cols]].to_numpy(float)
        equal=np.allclose(x,y,rtol=1e-9,atol=1e-9,equal_nan=True)
        rr.append(dict(site=site,n_condicoes=len(m),mesmas_condicoes=True,n_mudancas_candidato=int(m.candidato_novo.ne(m.candidato_historico).sum()),n_mudancas_avaliabilidade=int(m.avaliavel_novo.ne(m.avaliavel_historico).sum()),escores_equivalentes_tolerancia=equal,maior_diferenca_escores=float(np.nanmax(abs(x-y))),maior_diferenca_cosseno=float(abs(m.cosseno_min_novo-m.cosseno_min_historico).max())))
    save(pd.DataFrame(rr),out,'historico_vs_padronizado_condicoes')
    oo=csv(root/'historico/avaliacao_R4_historica.csv'); nn=csv(out/'tabelas/avaliacao_R4_reanalises_e_USC.csv')
    r=[]
    for site in ['IMTM','FMP']:
        z=nn[nn.site.eq(site)].merge(oo[oo.site.eq(site)],on=['composto','placa','dose_chave'],suffixes=('_novo','_historico'),validate='one_to_one')
        r.append(dict(site=site,n_candidatos=len(z),status_inalterados=bool(z.status_R4_novo.eq(z.status_R4_historico).all()),erro_max_A_R4=float(abs(z.A_Mito_R4_novo-z.A_Mito_R4_historico).max())))
    save(pd.DataFrame(r),out,'historico_vs_padronizado_R4')
    oo=csv(root/'historico/modelos_resumo.csv');nn=csv(out/'tabelas/modelos_resumo.csv')
    z=nn.merge(oo,on=['site','modelo'],suffixes=('_novo','_historico'),validate='one_to_one')
    for c in ['MAE_log','RMSE_log','R2_log','MAE_A','RMSE_A','R2_A']:z['delta_'+c]=z[c+'_novo']-z[c+'_historico']
    save(z,out,'historico_vs_padronizado_modelos')
    return pd.DataFrame(rr),pd.DataFrame(r),z

def supplemental_tables(root,out):
    info=[];sets={};inspection=[]
    cases=set(csv(out/'tabelas/casos_biologicos_quatro_locais.csv').composto)
    allc=csv(out/'tabelas/condicoes_quatro_locais.csv')
    for site in SITES:
        cat=csv(root/site/'04_catalogo_selecao.csv');selected=cat[boolean(cat.selecionada)];sets[site]=set(selected[selected.bloco.eq('Mito')].medida)
        wells=csv(root/site/'05_escores_por_poco.csv')
        info.append(dict(site=site,pocos_desenvolvimento=len(wells),medidas_iniciais=len(cat),selecionadas=len(selected),selecionadas_Mito=len(sets[site]),pocos_menos20=int(wells.n_objetos.lt(20).sum()),pct_menos20=100*wells.n_objetos.lt(20).mean()))
        coords=wells[wells.composto.isin(cases)].copy();coords['site']=site;coords['selecao_painel']='casos documentais previamente discutidos; sem selecao por acerto';inspection.append(coords)
    save(pd.DataFrame(info),out,'qualidade_e_selecao_por_local')
    shared=sorted(set.intersection(*sets.values()))
    save(pd.DataFrame({'medida':shared}),out,'medidas_Mito_selecionadas_comuns')
    # Availability only: no new target or score is computed using these common features.
    save(pd.concat(inspection,ignore_index=True),out,'pocos_casos_para_inspecao_R1_R3')
    dirs=csv(out/'tabelas/modelos_direcoes_recontadas.csv')
    a=[]
    for site,g in dirs.groupby('site'):
        p=g.pivot(index='direcao',columns='modelo',values='MAE_log')
        for m in p.columns.drop('XGBoost'):a.append(dict(site=site,comparador=m,direcoes_XGB_menor=int(p.XGBoost.lt(p[m]).sum()),n_direcoes=len(p),delta_medio=float((p.XGBoost-p[m]).mean())))
    save(pd.DataFrame(a),out,'ganhos_por_direcao_vs_todos')
    return shared

def run_standardized(root,out):
    root=Path(root);out=Path(out)
    result=comparison(root,out)
    protocol=audit_protocols(root,out)
    histc,histr,histm=compare_historical(root,out)
    shared=supplemental_tables(root,out)
    current=csv(out/'tabelas/modelos_resumo.csv')
    ex=csv(out/'tabelas/extremos_globais_sem_exclusao.csv');ex=ex[ex.modelo.eq('Ridge_medidas')]
    report=(out/'relatorio_comparativo.md').read_text()
    report+='\n## Padronizacao conferida\n\n'
    report+=f"Conferidos {protocol['parametros_comparados']} parametros cientificos: iguais nos quatro locais. As oito versoes declaradas tambem coincidem. Os hashes globais diferem porque as camadas de entrada e registro sao distintas; nao se afirma identidade binaria de notebooks inteiros. O nucleo das 14 celulas de calculo foi comparado nos cadernos disponibilizados.\n\n"
    report+=markdown(csv(out/'tabelas/versoes_declaradas_nas_execucoes.csv'))
    report+='\n\n## Historico versus reexecucao\n\n'+markdown(histc,digits=12)+'\n\n'+markdown(histr,digits=12)
    report+='\n\nOs resultados historicos nao foram substituidos. Modelos foram reajustados nas campanhas locais, com particoes e ambiente padronizados; estas tabelas nao isolam a contribuicao de cada mudanca. Nao escolher a versao pelo menor erro.\n\n'
    rr=histm[histm.site.isin(['IMTM','FMP'])&histm.modelo.isin(['XGBoost','Ridge_medidas'])]
    report+=markdown(rr[['site','modelo','MAE_log_historico','MAE_log_novo','MAE_A_historico','MAE_A_novo']])
    report+='\n\n## Extremos Ridge: diagnostico sem exclusao\n\n'+markdown(ex[['site','composto','direcao','fold','A_observado','A_predito','fracao_SAE_global']])
    report+='\n\nA fracao SAE usa todos os erros absolutos desse comparador no local (seis direcoes juntas), nao a media de fracoes dos folds. A reproducao do Ridge consta das auditorias locais salvas; nao foi recalculada a partir dos coeficientes nesta comparacao.\n\n'
    report+=f'## Proxima etapa biologica\n\nExistem {len(shared)} medidas mitocondriais selecionadas em comum. Essa lista identifica disponibilidade, nao estabelece comparabilidade de intensidade, aquisicao ou segmentacao; nenhum escore comum foi criado. O painel de casos inclui Alda-1, GLAUCINE, MM-102, TCID, HISPIDULIN e DHODH-IN-17 por criterios documentados anteriormente. As coordenadas dos pocos de desenvolvimento foram exportadas, mas nenhuma imagem foi aberta ou avaliada. As hipoteses biologicas do primeiro relatorio nao se tornaram demonstracoes mecanisticas com esta reexecucao.\n\n'
    report+='A analise de imagens deve incluir controles correspondentes, campos definidos por criterio uniforme e os exemplos discordantes. O escore quadratico elimina a direcao original das medidas; para interpretar reorganizacao, deve-se voltar aos valores com sinal e as imagens. A identidade farmacologica nao demonstra atividade do alvo na linhagem ou na dose usada. Nenhum enriquecimento, novo ajuste, transferencia de modelo ou avaliacao de MEDINA/R4 foi realizado.\n'
    (out/'relatorio_comparacao_padronizada.md').write_text(report,encoding='utf-8')
    result['protocol']=protocol;result['tests'].update(protocol);result['tests']['medidas_mito_comuns']=len(shared)
    (out/'conferencias.json').write_text(json.dumps(result['tests'],ensure_ascii=False,indent=2))
    check_sources(root)
    return result



## 4. Executar a comparacao

Uma divergencia de chave, referencia ou metrica interrompe a rotina. Um resultado nao avaliavel permanece distinto de um resultado negativo. Nenhum erro extremo e retirado para melhorar metricas.

In [ ]:
RESULTADO = run_standardized(FONTES, RUN / "resultados")
SAIDA = RESULTADO["out"]
display(RESULTADO["summary"])
display(RESULTADO["benefits"])
print(json.dumps(RESULTADO["tests"], ensure_ascii=False, indent=2))

## 5. Relatorio, mudancas e limites

A reexecucao pode preservar a descoberta e alterar predições de modelos ajustados em novas particoes. Essa diferenca nao isola um efeito exclusivo de semente ou versao. A escolha da campanha principal deve seguir a padronizacao, nao a selecao do resultado mais favoravel.

In [ ]:
display(Markdown((SAIDA / "relatorio_comparacao_padronizada.md").read_text(encoding="utf-8")))

## 6. Figuras

Os graficos sintetizam as tabelas auditadas. As linhas entre sitios nao sao trajetorias das mesmas celulas. Casos individuais sao apresentados como estudos descritivos, sem criar novas classes biologicas.

In [ ]:
LEGENDAS = json.loads((SAIDA / "legendas.json").read_text())
for imagem in sorted((SAIDA / "figuras").glob("*.png")):
    display(Image(filename=str(imagem)))
    display(Markdown(LEGENDAS[imagem.stem]))

## 7. Da comparacao a investigacao biologica

O arquivo `pocos_casos_para_inspecao_R1_R3.csv` localiza os pocos de Alda-1, GLAUCINE, MM-102, TCID, HISPIDULIN e DHODH-IN-17. Eles foram escolhidos no comparativo anterior por motivos documentados, incluindo discordancias. Nenhuma imagem foi inspecionada neste caderno.

Ha 114 nomes de medidas mitocondriais selecionadas em comum, mas nao foi construido um novo alvo com elas. Antes de interpretar distribuicao radial, textura ou intensidade como processos biologicos, precisamos comparar as medidas com sinal e a qualidade das imagens. Nao retirar um canal ou reler uma reserva para produzir mais candidatos.

In [ ]:
display(pd.read_csv(SAIDA / "tabelas/qualidade_e_selecao_por_local.csv"))
display(pd.read_csv(SAIDA / "tabelas/historico_vs_padronizado_condicoes.csv"))
display(pd.read_csv(SAIDA / "tabelas/extremos_globais_sem_exclusao.csv").query("modelo == 'Ridge_medidas'"))

## 8. Arquivar

A pasta e o ZIP preservam as tabelas, figuras, manifestos e versoes desta comparacao. Os modelos e as fontes originais permanecem em suas pastas. MEDINA/R4 continua sem avaliacao.

In [ ]:
from importlib.metadata import version
VERSOES_COMPARACAO = {k: version(k) for k in ["numpy", "pandas", "scipy", "matplotlib"]}
VERSOES_COMPARACAO["python"] = platform.python_version()
(SAIDA / "versoes_comparador.json").write_text(json.dumps(VERSOES_COMPARACAO, indent=2))
for nome in ["manifesto_fontes.json", "origens_execucoes.json", "auditoria_codigo_nucleos.json"]:
    shutil.copy2(FONTES / nome, SAIDA / nome)
PACOTE_RESULTADOS = shutil.make_archive(str(SAIDA), "zip", root_dir=SAIDA)
print("CONCLUIDO:", SAIDA)
print("PACOTE:", PACOTE_RESULTADOS)
print("Nenhum modelo treinado ou escore/candidato redefinido. MEDINA R4 nao lida.")